# 05 · 换地方:按**街道**取数据
想把整套流程搬到上海别的地方?**换地方 = 换一个街道。** 这本专讲这件事。

> 和新加坡 OSM 版最根本的不同:OSM 版用「中心点 + 半径」框一个**方块**;
> 上海版的研究单位是**街道**(乡镇街道层的真实多边形)——**非方形、大小不一**,
> 因为形态家族(里弄 / 工人新村 / CBD)本就是按街道沉淀的。按街道取,才对得上「谁更有权」的真实治理单位。

## 怎么执行
- 点一格,按 **Shift+Enter** 执行;或选单 Run → Run All Cells 从头跑。
- 结果与图会显示在该格下面。代码都在 `engine/` 文件夹,平时不用打开。

In [ ]:
# 让 notebook 找到 engine 里的代码(这格不用改,直接执行)
import sys, os
sys.path.insert(0, os.path.abspath("engine"))
import importlib, config; importlib.reload(config)
import common as C, plots
print("就绪 ·  当前站点 SLUG =", config.SLUG, "(", config.site_name(), ")")

## 一、随包就带 9 个街道(跨 3 个形态家族)—— 离线就能切
下面这些街道的缓存随包附带,**改 `config.SLUG` 就能立刻切换**,不需要数据集、不需要联网。

In [ ]:
import yaml
from pathlib import Path
print("随包带缓存的街道(改 config.SLUG 即可切):\n")
for s in config.SITES:
    p = C.DATA / s["slug"] / "buildings.parquet"
    mark = "✓缓存在" if p.exists() else "·需建"
    print("  %-10s %-7s %-12s %s" % (s["slug"], mark, s["name"], s["family"]))
print("\n默认主册/报告跑这 3 个代表:", config.REPORT_SITES)

## 二、换成已有缓存的街道(最简单)
**两种改法,效果一样:**
- 长期:打开 `config.py` 把 `SLUG = "..."` 改掉、存档,回到上面「准备」格重跑。
- 临时:在本 notebook 里直接设 `config.SLUG`(下面这格示范切到外滩),马上就能用。

In [ ]:
config.SLUG = "waitan"          # ✏️ 换成 config.SITES 里任一 slug(如 nanjingxi / pengpu / laoximen …)
df = C.current_buildings(config.SLUG)
df = C.assign_all(df)
h = df["height_m"].astype(float)
print("现在是:%s(%s)" % (config.site_name(), config.SLUG))
print("  %d 栋 · 最高 %.0fm · 中位 %.0fm" % (len(df), h.max(), h.median()))
print("  角色:", df.stakeholder.value_counts().to_dict())
# 接着就能跑主册 step1–5 / 进阶算子,全部对这个新街道生效。

## 三、取一个**全新**街道(这 9 个之外)—— 需要数据集
想要随包 9 个之外的街道(例如你研究的某个街道),就从原始「上海城市数据集」按**街道名**现建缓存。
需要先在 `config.py` 设 `DATASET_ROOT`(见 `数据集说明.md`)。下面这格:**列出数据集里所有街道名**,方便你挑。

In [ ]:
P = C.dataset_paths()
if P["JD"].exists():
    import geopandas as gpd
    jd = gpd.read_file(P["JD"])
    names = sorted(n for n in jd["name"].dropna().unique())
    print("数据集里共 %d 个乡镇街道。含『新村』的(工人新村家族)示例:" % len(names))
    print("  ", [n for n in names if "新村" in n][:12])
    print("含『路』的示例:", [n for n in names if n.endswith("路街道")][:8])
else:
    print("未设 DATASET_ROOT(或数据集不在)→ 这步需要数据集。")
    print("→ 没有数据集也没关系:上面随包 9 个街道足够练全套流程。")
    print("→ 想要别的街道:下载数据集、在 config.py 设 DATASET_ROOT,见 数据集说明.md。")

## 四、建这个新街道的缓存
选好街道名(乡镇街道层的精确名,支持「包含匹配」),给它一个英文 `slug`(输出标识),建缓存。
没有数据集时这格会自动跳过、提示用随包缓存。建好后,它就和随包街道一样可离线用。

In [ ]:
NEW_NAME = "天山路街道"      # ✏️ 你要的街道名(数据集里的精确名;可用包含匹配)
NEW_SLUG = "tianshan"        # ✏️ 给它一个英文 slug(决定 data/<slug>/ 与 out/<slug>/)
if config.DATASET_ROOT and C.dataset_paths()["AI"].exists():
    C.build_cache(NEW_NAME, NEW_SLUG)        # 裁切 + 多源 join → data/<slug>/buildings.parquet
    config.SLUG = NEW_SLUG
    print("建好并切到:", config.site_name(NEW_SLUG))
else:
    print("没有数据集 → 跳过建缓存。先用随包的 9 个街道即可(见第一格)。")

## 五、(可选)写进 config.py 长期用
临时改 `config.SLUG` 只在本次有效。想长期用、并让它进 `SITES` 菜单,在 `config.py` 的 `SITES` 加一行:
```python
SITES = [
    ...,
    {"slug": "tianshan", "name": "天山路街道", "family": "你给的家族标签"},
]
```
要让 `build_report.py` 也出它的报告,把 slug 加进 `REPORT_SITES`。

## 诚实边界 & 小结
- 研究单位是**街道**(行政/治理单位),非方框、非产权边界;一个街道一个 slug。
- 街道名用乡镇街道层的精确名(支持包含匹配,如 `subdistrict("天山")`)。
- 换地方后,回 [`03 主册`] / [`04 进阶`] 用新街道**重跑**即可——同一套权力逻辑,长在不同的真实基底上。
> 这正是进阶册的点题:**权力的几何逻辑对基底不变,但结果在乎基底**。换地方,就是去验证这句话。